In [1]:
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sms
import statsmodels.formula.api as smf

from rubins import (_pool_est,
                    _pool_variance,
                    _calc_fmi_rvi,
                    _pool_dof,
                    )


In [18]:
def _fisher_z(r2, num_obs):
    """
    Performs a fisher's Z transform for R2 correlation data

    Parameters
    ----------
    r2: pd.Series
        The correlation coeffecients for the analysis
    num_obs: int
        The number of observations in the orginal data set

    Returns
    pd.Series
        The z-transformed correlation coeffecient
    pd.Series
        The standard deviation in the z-transformed correlation coeffecient

    References
    ----------
    [1] Harel (2009). "The estimation of R2 and adjusted R2 in incomplete 
        data sets using multiple imputation". Journal of Applied Statitics. 
        36:1109. doi: 10.1080/02664760802553000
    [2] Fisher Z transform. (2 June 2024) In Wikipedia.
        https://en.wikipedia.org/wiki/Fisher_transformation 

    Also See
    --------
    _throwback_z
    """
    ra = np.sqrt(r2)
    q = 0.5 * np.log((1 + ra) / (1 - ra))
    v = np.square(1 / np.sqrt(num_obs - 3))

    return q, v

In [19]:
def _throwback_z(q):
    """
    Undoes the Fisher's Z transformation

    Parameters
    ----------
    q: pd.Series
        The fisher z-transforemd correlation coeffecient
    
    Returns
    -------
    pd.Series
        The untransfromed correlation coeffecient values    

    References
    ----------
    [1] Harel (2009). "The estimation of R2 and adjusted R2 in incomplete 
        data sets using multiple imputation". Journal of Applied Statitics. 
        36:1109. doi: 10.1080/02664760802553000
    [2] Fisher Z transform. (2 June 2024) In Wikipedia.
        https://en.wikipedia.org/wiki/Fisher_transformation 

    Also See
    --------
    _fisher_z
    """
    ra = np.tanh(q)
    r2 = np.square(ra)

    return r2

In [20]:
name_col = 'name',
order_col = 'var_order'
impute_col = 'imputation',
idx_cols = ['name', 'var_order']

In [21]:
adonis = pd.read_csv('tests/test_res.tsv', sep='\t')
adonis.set_index(list(np.hstack([idx_cols, impute_col])), 
                 inplace=True)
adonis

,,,df,SumOfSquares,r2,stat,pval
name,var_order,imputation,,,,,
age,1.0,1,1,0.611764,0.008010,2.074553,0.004
ethnicity,2.0,1,1,1.948138,0.025506,6.606330,0.001
bmi_class,3.0,1,2,0.920349,0.012050,1.560497,0.007
alcohol_use,4.0,1,3,0.916780,0.012003,1.036298,0.348
birth_country,5.0,1,3,1.503624,0.019686,1.699646,0.001
...,...,...,...,...,...,...,...
bmi_class,3.0,10,2,0.967500,0.012667,1.640500,0.001
alcohol_use,4.0,10,3,0.887293,0.011617,1.003000,0.448
birth_country,5.0,10,3,1.464109,0.019169,1.655036,0.001


In [22]:
df_ = adonis.xs('Total', level='name', axis='index')['df'] + 1
df_

var_order  imputation
NaN        1             250
           2             250
           3             250
           4             250
           5             250
           6             250
           7             250
           8             250
           9             250
           10            250
Name: df, dtype: int64

In [23]:
def _extract_num_obs(adonis, name_col):
    """
    Gets the number of 
    """
    df_ = df_ = adonis.xs('Total', level='name', axis='index')['df'] + 1
    num_obs = pd.Series([df_.drop_duplicates().values[0]] * len(adonis),
                        index=adonis.index,
                        name='num_obs')
    return num_obs
                        
    
num_obs = _extract_num_obs(adonis, name_col)

In [24]:
def _pool_r2(adonis, num_obs, name_col='name', imput_col='imputation',
             index_cols=['name', 'var_order'], r2_col='r2'):
    """
    Calculates the pooled R2 value
    
    """
    # Casts the data to a correlation coeffecient and gets the parameter
    # variance
    q_long, v_long = _fisher_z(adonis[r2_col].drop(['Total'], level=name_col),
                               num_obs)
    # Pivots the variable to longer so we have each column as an imputed
    # transformed value
    q_wide = q_long.reset_index().pivot(index=index_cols,
                                        columns=impute_col,
                                        values=r2_col)
    v_wide = v_long.reset_index().pivot(index=index_cols,
                                        columns=imput_col,
                                        values=v_long.name)

    # Calculates the pooled 

In [26]:
adonis_q, adonis_z = _fisher_z(adonis['r2'].drop('Total', level='name'), 
                               num_obs.drop('Total', level='name'))

In [30]:
q_wide = adonis_q.reset_index().pivot(index=idx_cols,
                                      columns='imputation',
                                      values='r2')

z_wide = adonis_z.reset_index().pivot(index=idx_cols,
                                      columns='imputation',
                                      values='num_obs')

In [31]:
q_wide.loc['age'].values

array([[0.08973622, 0.08796234, 0.08785097, 0.09045728, 0.09035581,
        0.08890988, 0.08730415, 0.08727498, 0.08937271, 0.08872715]])

In [32]:
m = len(q_wide.columns)

In [33]:
q_pooled = _pool_est(q_wide)
q_pooled

name           var_order
Residual       6.0          1.954278
age            1.0          0.088795
alcohol_use    4.0          0.109652
birth_country  5.0          0.139095
bmi_class      3.0          0.112513
ethnicity      2.0          0.161759
dtype: float64

In [34]:
z_wide.loc['age'].values

array([[0.00404858, 0.00404858, 0.00404858, 0.00404858, 0.00404858,
        0.00404858, 0.00404858, 0.00404858, 0.00404858, 0.00404858]])

In [35]:
def _pool_variance_simp(params, variance, m):
    """
    Pools the variance estimates, not variance/covariance
    """
    var_w = variance.mean(axis=1)

    coef_pool = _pool_est(params).to_frame().values
    vcov_b = np.dot((params - coef_pool), (params - coef_pool).T) / (m-1)
    var_b = pd.Series(np.diag(vcov_b), index=var_w.index)
    var_f = var_b / m
    var_t = var_b + var_w + var_f

    return var_w, var_b, var_f, var_t

In [36]:
def _pool_r2(adonis_df, num_obs, alpha=0.05, name_col='name', 
             imput_col='imputation',
             index_cols=['name', 'var_order'], r2_col='r2'):
    """
    Pools correlation coeffecients 

    Parameters
    ----------
    adonis_df: DataFrame
        A dataframe with a multi level index including the adonis results.
        The frame should be indexed by a multiple level index whcih is 
        based on the `index_col` plus the `imput_col`, where the 
        variable name is in the `name_col` and the imputation identifier 
        is in the `input_col` level.
        The dataframe should also include the `r2_col` whcih contains the
        correlation coeffecient column.
    num_obs: int
        The number of observations in the original data set
    alpha : float, (0, 1)
        The critical value for the 95% confidence interval
    name_col, imput_col: str
        The level names identifying the variable name and imputation id,
        respectively, in the the adonis_df index
    index_col: list
        The list of level names (sans `imputation_col`) which identify
        a unique variable in the adonis_df dataframe. This likely contains
        the `name_col` but may contain other variables.
    r2_col: str
        The name of the column containing the adonis R2 statistic.

    Returns
    -------
    DataFrame
        A dataframe with a multi level index based on `index_cols` which
        summarises the results of the pooling. Output columns are:
            r2 - The pooled R2 estimate for the data
            r2_ci_lo - The lower bound of the pooled R2 value
            r2_ci_hi - The upper bound of the pooled R2 value
            `fmi`: The Fraction of Missing data estimate (uncorrected)
            `rvi`: The relative variance increase due ot missingness

    Also See
    --------
    pool_fits


    References
    ----------
    [1] Harel (2009). "The estimation of R2 and adjusted R2 in incomplete 
        data sets using multiple imputation". Journal of Applied Statitics. 
        36:1109. doi: 10.1080/02664760802553000
    """

    # Casts the data into a correlation matrix and gets the variance
    # in that parameter, based on a fisher Z transform
    q_long, v_long = \
        _fisher_z(adonis[r2_col].drop(['Total'], level=name_col), num_obs)

    # Pivots the variable to longer so we have each column as an imputed
    # transformed value
    q_wide = q_long.reset_index().pivot(index=index_cols,
                                        columns=impute_col,
                                        values=r2_col)
    v_wide = v_long.reset_index().pivot(index=index_cols,
                                        columns=imput_col,
                                        values=v_long.name)
    m = len(q_wide.columns)
    
    # Calculates the pooled q estimate
    q_pool = _pool_est(q_wide)

    # Calculates the pooled variance estimate
    var_w, var_b, var_f, var_t = _pool_variance_simp(q_wide, v_wide, m)
    bse_t = np.sqrt(var_t)

    # Gets the fmi and rvi
    fmi, rvi = _calc_fmi_rvi(var_b, var_f, var_t, var_w, m)

    # Calculates the CI on the q-value
    q_width = scipy.stats.norm.ppf(1 - alpha / 2) * bse_t
    q_ci_lo = q_pool - q_width
    q_ci_hi = q_pool + q_width

    # Summarizes the results
    r2_summary = pd.DataFrame({
        'r2': _throwback_z(q_pool),
        'r2_ci_lo':  _throwback_z(q_ci_lo),
        'r2_ci_hi': _throwback_z(q_ci_hi),
        'fmi': fmi,
        'rvi': rvi,
    })

    return r2_summary

In [39]:
res = _pool_r2(adonis, num_obs.drop(['Total'], level='name'))

In [43]:
res

,,r2,r2_ci_lo,r2_ci_hi,fmi,rvi
name,var_order,,,,,
Residual,6.0,0.922850,0.902064,0.939371,0.002205,0.002206
age,1.0,0.007843,0.001290,0.044243,0.000379,0.000379
alcohol_use,4.0,0.011928,0.000227,0.052984,0.000312,0.000312
birth_country,5.0,0.019101,0.000206,0.066503,0.000538,0.000538
bmi_class,3.0,0.012553,0.000151,0.054262,0.001228,0.001229
ethnicity,2.0,0.025716,0.001368,0.077798,0.000730,0.000730


In [132]:
var_b

name           var_order
Residual       NaN          0.000073
age            1.0          0.000013
alcohol_use    4.0          0.000010
birth_country  5.0          0.000018
bmi_class      3.0          0.000041
ethnicity      2.0          0.000024
dtype: float64

In [135]:
a = np.array([[0.08973622, 0.08796234, 0.08785097, 0.09045728, 
                            0.09035581, 0.08890988, 0.08730415, 0.08727498, 
                            0.08937271, 0.08872715]])

In [139]:
var_b2 = np.diag(np.dot((a - a.mean()), (a - a.mean()).T) / (10 - 1))

In [141]:
t_ = var_w + var_b2 + var_b2/10

In [143]:
t_.loc['age'].values

array([0.00405012])